<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Chatbot%20K%C3%A9sz%C3%ADt%C3%A9se%20NLP%20Seg%C3%ADts%C3%A9g%C3%A9vel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chatbot Készítése NLP Segítségével

**Licenc: CC BY-NC-SA 4.0**

A **chatbotok** interaktív rendszerek, amelyek természetes nyelven kommunikálnak a felhasználókkal. Különböző megközelítésekkel építhetők: szabály-alapú, retrieval és generatív.

## Chatbot típusok és jellemzőik

| Típus | Működés | Előny | Hátrány |
|-------|---------|-------|----------|
| **Rule-based** | If-then szabályok | Egyszerű, kontrollált, prediktábilis | Nem skálázható, merev |
| **Retrieval** | Leghasonlóbb válasz keresése | Konzisztens, nem hallucinal | Fix válaszkészlet |
| **Generative** | Válasz generálása | Flexibilis, kreatív | Hallucináció, kontroll hiány |
| **Hybrid** | Több megközelítés kombinációja | Kiegyensúlyozott | Komplexebb |

## Használati esetek

- **Ügyfélszolgálat**: FAQ válaszok, jegyfoglalás
- **E-commerce**: Termékajánlás, rendelés követés
- **Egészségügy**: Triage, időpontfoglalás
- **Oktatás**: Tutoring, kérdés-válasz

## Tartalomjegyzék
1. Rule-based chatbot
2. Retrieval-based chatbot
3. Seq2Seq generatív chatbot
4. LLM-alapú chatbot
5. Conversation management

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import re

torch.manual_seed(42)

## 1. Rule-based chatbot

A legegyszerűbb megközelítés: **regex pattern matching** és fix válaszok.

### Működés

```
User input → Pattern matching → Ha illeszkedik → Fix válasz
                              → Ha nem → Default válasz
```

### Mikor használd?

- ✅ Jól definiált, korlátozott domain (pl. FAQ)
- ✅ Prediktábilis válaszok kellenek
- ✅ Nincs ML infrastruktúra
- ❌ Nyílt domain beszélgetések
- ❌ Sok variáció a kérdésekben

In [ ]:
class RuleBasedBot:
    def __init__(self):
        self.rules = [
            (r'szia|hello|hali', 'Szia! Miben segíthetek?'),
            (r'hogy.*vagy', 'Köszönöm, jól vagyok!'),
            (r'mi.*neved', 'A nevem ChatBot.'),
            (r'időjárás', 'Sajnos nem tudok időjárást nézni.'),
            (r'viszlát|bye', 'Viszlát! Szép napot!'),
        ]
        self.default = 'Nem értem, kérlek fogalmazd át!'

    def respond(self, text):
        text = text.lower()
        for pattern, response in self.rules:
            if re.search(pattern, text):
                return response
        return self.default

bot = RuleBasedBot()
for msg in ['Szia!', 'Hogy vagy?', 'Mi a neved?', 'Viszlát']:
    print(f"User: {msg}\nBot: {bot.respond(msg)}\n")

## 2. Retrieval-based chatbot

A retrieval megközelítés egy **előre definiált válaszkészletből** választja ki a legjobban illeszkedőt.

### Működés

1. **Kérdés embedding**: User input → vektor
2. **Hasonlóság számítás**: Cosine similarity minden tárolt kérdéssel
3. **Top-k retrieval**: Leghasonlóbb kérdés(ek) kiválasztása
4. **Válasz visszaadása**: A hozzá tartozó válasz

### Embedding opciók

| Módszer | Komplexitás | Minőség |
|---------|-------------|---------|
| **BoW/TF-IDF** | Alacsony | Közepes |
| **Word2Vec átlag** | Közepes | Jó |
| **Sentence-BERT** | Magas | Kiváló |

### Előnyök

- Nem hallucinal - csak "ismert" válaszok
- Könnyű frissíteni (új Q-A párok hozzáadása)
- Gyors inference

In [ ]:
class RetrievalBot:
    def __init__(self, qa_pairs):
        self.questions = [q for q, a in qa_pairs]
        self.answers = [a for q, a in qa_pairs]
        # Egyszerű BoW embedding
        self.vocab = set(w for q in self.questions for w in q.lower().split())
        self.vocab = {w: i for i, w in enumerate(self.vocab)}

    def embed(self, text):
        vec = np.zeros(len(self.vocab))
        for w in text.lower().split():
            if w in self.vocab:
                vec[self.vocab[w]] = 1
        return vec / (np.linalg.norm(vec) + 1e-8)

    def respond(self, query):
        q_emb = self.embed(query)
        best_idx, best_sim = 0, -1
        for i, q in enumerate(self.questions):
            sim = np.dot(q_emb, self.embed(q))
            if sim > best_sim:
                best_sim, best_idx = sim, i
        return self.answers[best_idx] if best_sim > 0.3 else "Nem találtam releváns választ."

qa_pairs = [
    ("Mikor nyitva az iroda", "Hétfőtől péntekig 9-17 óráig."),
    ("Hogy tudok fizetni", "Bankkártyával vagy készpénzzel."),
    ("Hol találom a terméket", "A webshopban vagy üzletünkben."),
]

bot = RetrievalBot(qa_pairs)
print(f"User: Mikor van nyitva?\nBot: {bot.respond('Mikor van nyitva?')}")

## 3. Seq2Seq generatív chatbot

A **generatív chatbot** a választ tokenről tokenre generálja, nem egy fix készletből választ.

### Seq2Seq architektúra

```
Encoder: User input → Hidden state (kontextus)
                         ↓
Decoder: Hidden state → Generated response
```

### Komponensek

- **Encoder**: RNN/LSTM/GRU - input feldolgozása
- **Decoder**: RNN/LSTM/GRU - válasz generálása autoregresszíven
- **Attention**: Encoder hidden state-ekre figyelés (opcionális, de ajánlott)

### Teacher forcing

Tanításnál a decoder inputja a **ground truth** előző token (nem a saját predikciója). Ez stabilizálja a tanulást, de "exposure bias"-t okozhat.

In [ ]:
class Seq2SeqBot(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.encoder = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.decoder = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, src, tgt):
        _, hidden = self.encoder(self.embedding(src))
        dec_out, _ = self.decoder(self.embedding(tgt), hidden)
        return self.output(dec_out)

model = Seq2SeqBot(vocab_size=1000)
src = torch.randint(0, 1000, (2, 10))
tgt = torch.randint(0, 1000, (2, 8))
print(f"Output: {model(src, tgt).shape}")

## 4. LLM-alapú chatbot

A modern megközelítés: **előtanított LLM** (GPT-4, Claude, LLaMA) használata API-n vagy lokálisan.

### Előnyök

- Nincs szükség saját training-re
- Kiváló nyelvértés és generálás
- Zero-shot és few-shot képességek

### System prompt

A chatbot viselkedését a **system prompt** határozza meg:

```
Te egy barátságos ügyfélszolgálatos vagy a TechShop-nál.
- Mindig udvariasan válaszolj
- Ha nem tudod a választ, ajánld fel, hogy kapcsolod az ügyfelet egy kollégához
- Soha ne adj pénzügyi tanácsot
```

### Conversation history

Az LLM-ek **stateless**-ek - a teljes beszélgetés történetet el kell küldeni minden kéréssel!

In [ ]:
print("""
from openai import OpenAI

client = OpenAI()
messages = [{"role": "system", "content": "Te egy segítőkész ügyfélszolgálatos vagy."}]

while True:
    user_input = input("User: ")
    if user_input.lower() == 'exit':
        break

    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="gpt-4",
        messages=messages
    )

    assistant_msg = response.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_msg})
    print(f"Bot: {assistant_msg}")
""")

## 5. Conversation management

A chatbot működéséhez több komponens kell a válaszgeneráláson kívül.

### Fontos elemek

| Komponens | Funkció |
|-----------|---------|
| **Intent detection** | Mit akar a user? (pl. "rendelés", "panasz") |
| **Entity extraction** | Releváns információk (dátum, termék neve) |
| **State management** | Hol tartunk a beszélgetésben? |
| **Fallback handling** | Mit tegyünk, ha nem értjük? |
| **Handoff** | Mikor kell ember? |

### Conversation flow példa

```
User: "Szeretnék rendelni"
  → Intent: ORDER
  → Bot: "Mit szeretne rendelni?"

User: "2 pizza"  
  → Intent: ORDER, Entity: {item: "pizza", quantity: 2}
  → Bot: "2 pizzát. Milyen címre szállítsuk?"

User: "Budapest, Fő utca 1."
  → Entity: {address: "Budapest, Fő utca 1."}
  → Bot: "Rendben, 2 pizza a Budapest, Fő utca 1. címre. Megerősíti?"
```

## Összefoglalás

### Megközelítések összehasonlítása

| Megközelítés | Mikor használd | Példa use case |
|--------------|----------------|----------------|
| **Rule-based** | Egyszerű, fix válaszok | FAQ bot, üdvözlés |
| **Retrieval** | Fix válaszkészlet, nem hallucinal | Ügyfélszolgálat |
| **Seq2Seq** | Nyílt domain, kontrollált | Speciális domain |
| **LLM-alapú** | Általános, flexibilis | ChatGPT-szerű asszisztens |

### Best practices

1. **Kezdd egyszerűen**: Rule-based vagy retrieval
2. **Mérj**: User satisfaction, task completion rate
3. **Fallback**: Mindig legyen "nem értem" kezelés
4. **Human handoff**: Tudjon embert hívni, ha kell
5. **Monitoring**: Logold a beszélgetéseket (privacy!)

### Evaluation metrikák

| Metrika | Mit mér |
|---------|---------|
| **Task completion** | Sikeresen megoldotta a feladatot? |
| **User satisfaction** | CSAT, NPS |
| **Response time** | Válaszidő |
| **Fallback rate** | Hányszor nem értette? |

### Következő lépések

A következő notebookban a **Fine-Tuning és LoRA** technikákat nézzük, amivel az LLM-eket saját feladatra szabhatjuk!